# GANS -- Generator vs Discriminator

Goodfellow's trick in 2014 was to skip density entirely. Two networks, one makes fakes. One catches them. They fight until the fakes are indistinguishable from the real. 

It shouldn't work. It often doesn't. When it does, the samples are still the sharpest in the literature for narrow domains.

## Problem definition

VAEs produce blurry samples because their MSE decoder loss in Bayes-optimal for mean image -- and the mean of many plausible digits is a fuzzy digit. You want a loss that rewards plausibility, not pixel-wise proximity to any one target. There is not closed-form for plausibility. You have to learn it.

Train a classifier `D(x)` to distinguish real imanges from fakes. Train a generator `G(z)` to fool `D`. The loss signal for `G` is whatever `D` currently thinks, make something look real. That signal updates as `G` improves, chasing a moving target. If both networks converge, `G` has learned the data distribution without ever writing down `log p(x)`.

```
min_G max_D  E_real[log D(x)] + E_fake[log(1 - D(G(z)))]
```

## Basic Concept

GAN traing: generator and discriminator in minimax.

### Generator `G(z)`

Maps a noise vector `z ~ N(0, I)` to a sample `x_hat`. A decoder-shaped network (dense or transposed conv)

### Discriminator `D(x)`

Maps a sample to scalar probability (or score). Real -> 1, Fake -> 0

### Loss

Tow alternating updates:
1. Train D: `loss_D = -[log D(x) + log (1 - D(G(z)))]`. Binary cross entropy on real = 1, fake = 0.
2. Train G: `loss_G = -log D(G(z))` This is non-saturating from Goodfellow used, if D is confident, original log (1 - D(G(z))) kills gradient.

## Traing Loop

One step of `D`, one step of `G`, repeat.

### Why it works.

If `G` perfectly matches `p_data`, then `D` cannot do better than chance and output 0.5 everywhere. G gets more gradient.

### Why it breaks.

Mode collapse (`G` finds one mode `D` can't classify and mints it forever); vanishing gradient (`D` learns too fast and log `D` saturates); Training instability.

# Build your Own

In [4]:
import torch
import torch.nn as nn
import random

torch.manual_seed(0)
random.seed(0)


class Generator(nn.Module):
    def __init__(self, z_dim, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden, hidden),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden, 1),
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden, hidden),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x)


# 1D 双峰数据 → z_dim=1 足够；4 维噪声反而难训、易 mode collapse
z_dim, hidden = 1, 64

G = Generator(z_dim, hidden)
D = Discriminator(hidden)

batch = 128
G_Opt = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
D_Opt = torch.optim.Adam(D.parameters(), lr=1e-4, betas=(0.5, 0.999))
criterion = nn.BCEWithLogitsLoss()


def sample_real(n):
    out = []
    for _ in range(n):
        if random.random() < 0.5:
            out.append([random.gauss(-2.0, 0.4)])
        else:
            out.append([random.gauss(2.0, 0.4)])
    return torch.tensor(out, dtype=torch.float32)


for step in range(1, 5001):
    reals = sample_real(batch)
    noise = torch.randn(batch, z_dim)

    D.train()
    D_Opt.zero_grad(set_to_none=True)
    real_logits = D(reals)
    with torch.no_grad():
        fakes_for_d = G(noise)
    fake_logits = D(fakes_for_d.detach())
    loss_D = criterion(real_logits, torch.ones_like(real_logits)) + criterion(
        fake_logits, torch.zeros_like(fake_logits)
    )
    loss_D.backward()
    D_Opt.step()

    for _ in range(2):
        noise = torch.randn(batch, z_dim)
        G.train()
        G_Opt.zero_grad(set_to_none=True)
        fake_logits = D(G(noise))
        loss_G = criterion(fake_logits, torch.ones_like(fake_logits))
        loss_G.backward()
        G_Opt.step()

    if step % 500 == 0:
        with torch.no_grad():
            probe = G(torch.randn(500, z_dim)).squeeze()
            p_real = torch.sigmoid(D(sample_real(500))).mean().item()
            p_fake = torch.sigmoid(D(G(torch.randn(500, z_dim)))).mean().item()
            near_neg2 = ((probe > -3) & (probe < -1)).float().mean().item()
            near_pos2 = ((probe > 1) & (probe < 3)).float().mean().item()
        print(
            f"step {step}: loss_D {loss_D.item():.3f}, loss_G {loss_G.item():.3f} | "
            f"D(real)={p_real:.2f} D(fake)={p_fake:.2f} | "
            f"near -2={near_neg2:.0%} near +2={near_pos2:.0%}"
        )





step 200: loss_D 1.357, loss_G 0.751 | D(real)=0.48 D(fake)=0.47 | modes left=0% right=100%
step 400: loss_D 1.399, loss_G 0.721 | D(real)=0.49 D(fake)=0.49 | modes left=0% right=100%
step 600: loss_D 1.240, loss_G 0.915 | D(real)=0.48 D(fake)=0.40 | modes left=0% right=100%
step 800: loss_D 1.190, loss_G 0.890 | D(real)=0.53 D(fake)=0.41 | modes left=0% right=100%
step 1000: loss_D 1.081, loss_G 0.855 | D(real)=0.57 D(fake)=0.43 | modes left=0% right=100%
step 1200: loss_D 1.070, loss_G 0.894 | D(real)=0.60 D(fake)=0.41 | modes left=1% right=99%
step 1400: loss_D 1.279, loss_G 0.745 | D(real)=0.61 D(fake)=0.49 | modes left=36% right=64%
step 1600: loss_D 1.631, loss_G 0.568 | D(real)=0.55 D(fake)=0.58 | modes left=52% right=48%
step 1800: loss_D 1.461, loss_G 0.686 | D(real)=0.46 D(fake)=0.50 | modes left=55% right=45%
step 2000: loss_D 1.350, loss_G 0.909 | D(real)=0.44 D(fake)=0.40 | modes left=60% right=40%


In [5]:
import sys
from pathlib import Path

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import SectionPrinter

import matplotlib.pyplot as plt
import numpy as np

G.eval()
with torch.no_grad():
    reals = sample_real(2000).squeeze().numpy()
    fakes = G(torch.randn(2000, z_dim)).squeeze().numpy()

with SectionPrinter("Real vs generated"):
    fig, ax = plt.subplots(figsize=(8, 4))
    bins = np.linspace(-4, 4, 40)
    ax.hist(reals, bins=bins, alpha=0.5, density=True, label="real data")
    ax.hist(fakes, bins=bins, alpha=0.5, density=True, label="G(z)")
    ax.axvline(-2, color="gray", ls="--", lw=1)
    ax.axvline(2, color="gray", ls="--", lw=1)
    ax.set_xlabel("x")
    ax.set_ylabel("density")
    ax.set_title("Target: two bumps at -2 and +2")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print("sample G(z):", np.round(fakes[:10], 2))

tensor([1.4371], grad_fn=<ViewBackward0>)
tensor([2.9076], grad_fn=<ViewBackward0>)
tensor([0.2448], grad_fn=<ViewBackward0>)
tensor([-3.9434], grad_fn=<ViewBackward0>)
tensor([-2.6386], grad_fn=<ViewBackward0>)
tensor([-2.0910], grad_fn=<ViewBackward0>)
tensor([-1.5838], grad_fn=<ViewBackward0>)
tensor([-7.4434], grad_fn=<ViewBackward0>)
tensor([2.0341], grad_fn=<ViewBackward0>)
tensor([-2.7184], grad_fn=<ViewBackward0>)
